# HPD 5 — Entregable: Sistema RAG con evaluación

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/evaluables/hpd5-entregable.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> **Peso en la nota:** 7.5 %
>
> **Plazo:** 7 días tras la sesión presencial de HPD 5.
>
> **Prerrequisito:** Haber realizado el [HPD 5 — RAG
> mixto](hpd5-rag-mixto.qmd) (taller presencial).
>
> **Entrega:** Notebook `.ipynb` ejecutado con todas las celdas
> completas. Cada ejercicio especifica qué variable debe contener el
> resultado para la corrección automática.

> **Cómo se corrige**
>
> Cada ejercicio pide que asignes el resultado a una variable con un
> nombre concreto (`eval1_dataset`, `eval2_ragas`, `eval3_iteracion`,
> `eval4_informe`). El script de corrección ejecutará tu notebook e
> inspeccionará esas variables.

## Setup

In [1]:
!pip install -q chromadb==1.5.9 sentence-transformers==5.5.1 langchain==1.3.2 langchain-community==0.3.31 langchain-text-splitters==1.1.2 pypdf==6.12.2 ragas==0.4.3 datasets==4.8.5 fpdf2==2.8.7 "pandas<3" matplotlib scikit-learn python-dotenv==1.2.2 openai==2.38.0 langchain-core==1.4.0 langgraph==1.2.2

In [2]:
import os, tempfile, shutil, sqlite3
"""
Setup compartido para la API key de Llamus en todos los notebooks.
Inyectado en cada .qmd/.ipynb vía scripts/sync_setup_keys.py (pre-render).

Orden de resolución:
  1. .env local (python-dotenv)
  2. os.environ ya configurado externamente
  3. Google Colab userdata: secret "LLAMUS_API_KEY" (preferente) o "LLM_API_KEY"

Las celdas posteriores pueden leer la clave con os.environ.get("LLM_API_KEY").
"""

import os

def _load_llm_api_key():
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    if not os.getenv("LLM_API_KEY"):
        try:
            from google.colab import userdata
            for name in ("LLAMUS_API_KEY", "LLM_API_KEY"):
                try:
                    value = userdata.get(name)
                    if value:
                        os.environ["LLM_API_KEY"] = value
                        break
                except Exception:
                    continue
        except ImportError:
            pass

_load_llm_api_key()

if os.getenv("LLM_API_KEY"):
    print("✓ LLM_API_KEY cargada.")
else:
    print("⚠️  LLM_API_KEY no encontrada.")
    print("    • Local: crea un archivo .env con LLM_API_KEY=tu_clave")
    print("    • Colab: añade un Secret 🔑 con nombre 'LLAMUS_API_KEY'")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fpdf import FPDF
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from langchain_openai import ChatOpenAI

plt.rcParams["figure.dpi"] = 100
print("Librerías cargadas correctamente.")

✓ LLM_API_KEY cargada.
Librerías cargadas correctamente.

In [3]:
LLM_KEY = os.getenv("LLM_API_KEY")
LLM_URL = os.getenv("LLM_URL", "https://llamus.cs.us.es/api/v1")

if not LLM_KEY:
    print("⚠️  Crea un archivo .env con:\n    LLM_API_KEY=tu_api_key")
else:
    llm = ChatOpenAI(model=os.getenv("LLM_MODEL", "gemma4:26b"), base_url=LLM_URL, api_key=LLM_KEY, temperature=0, max_tokens=2048)
    juez_llm = ChatOpenAI(model=os.getenv("LLM_JUDGE_MODEL", "gemma3:12b"), base_url=LLM_URL, api_key=LLM_KEY, temperature=0, request_timeout=60, max_retries=3)
    print(f"✅ LLM configurado ({os.getenv('LLM_MODEL', 'gemma4:26b')}).")
    print(f"✅ Juez de evaluación configurado ({os.getenv('LLM_JUDGE_MODEL', 'gemma3:12b')}).")

✅ LLM configurado (gemma4:26b).
✅ Juez de evaluación configurado (gemma3:12b).

In [4]:
MODEL_NAME = "intfloat/e5-small-v2"
CHUNK_SIZE = 400
CHUNK_OVERLAP = 60

model = SentenceTransformer(MODEL_NAME)
client = chromadb.EphemeralClient()

BASE_DIR = tempfile.mkdtemp()
CORPUS_DIR = os.path.join(BASE_DIR, "corpus")
os.makedirs(CORPUS_DIR, exist_ok=True)

print(f"Modelo: {MODEL_NAME} ({model.get_embedding_dimension()} dims)")
print(f"Directorio temporal: {CORPUS_DIR}")

Modelo: intfloat/e5-small-v2 (384 dims)
Directorio temporal: /var/folders/5q/dq_s1n552d37rqhcc02ppsn00000gn/T/tmpvz8cwurh/corpus

### Funciones auxiliares

In [5]:
class PDF(FPDF):
    pass

def crear_pdf(nombre, titulo, contenido):
    pdf = PDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, titulo.replace("—", "-").replace("€", "EUR"), new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)
    pdf.set_font("Helvetica", "", 11)
    pdf.multi_cell(0, 6, contenido.replace("—", "-").replace("€", "EUR"))
    pdf.output(os.path.join(CORPUS_DIR, nombre))

def cargar_y_trocear_pdfs(dir_path, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    documentos = []
    for filename in sorted(os.listdir(dir_path)):
        if not filename.endswith(".pdf"):
            continue
        loader = PyPDFLoader(os.path.join(dir_path, filename))
        documentos.extend(loader.load())
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return splitter.split_documents(documentos)

def indexar_chunks(collection, chunks, prefix=""):
    for i, chunk in enumerate(chunks):
        embedding = model.encode(chunk.page_content).tolist()
        fuente = os.path.basename(chunk.metadata.get("source", ""))
        cid = f"{prefix}{i:04d}"
        collection.add(
            ids=[cid],
            embeddings=[embedding],
            documents=[chunk.page_content],
            metadatas=[{"fuente": fuente, "idx": i}]
        )

def indexar_textos(collection, textos, metadatas_list, prefix=""):
    for i, texto in enumerate(textos):
        embedding = model.encode(texto).tolist()
        cid = f"{prefix}{i:04d}"
        meta = metadatas_list[i] if metadatas_list else {}
        collection.add(
            ids=[cid],
            embeddings=[embedding],
            documents=[texto],
            metadatas=[meta]
        )

def buscar(collection, query, k=5):
    qe = model.encode(query).tolist()
    res = collection.query(query_embeddings=[qe], n_results=k)
    resultados = []
    if res["ids"] and res["ids"][0]:
        for i in range(len(res["ids"][0])):
            resultados.append({
                "id": res["ids"][0][i],
                "texto": res["documents"][0][i],
                "distancia": res["distances"][0][i],
                "metadata": res["metadatas"][0][i] if res["metadatas"] else {},
            })
    return resultados

def rag(pregunta, collection_pdf, collection_sql, k_pdf=3, k_sql=3):
    if not LLM_KEY:
        return {"respuesta": "⚠️ Configura LLM_API_KEY en .env", "fuentes": [], "pregunta": pregunta}
    resultados_pdf = buscar(collection_pdf, pregunta, k=k_pdf)
    resultados_sql = buscar(collection_sql, pregunta, k=k_sql)
    combinados = sorted(resultados_pdf + resultados_sql, key=lambda r: r["distancia"])[:3]
    contexto = "\n\n".join(r["texto"] for r in combinados)
    prompt = f"""Eres un analista de negocio que responde preguntas basándote EXCLUSIVAMENTE en los fragmentos proporcionados.

Contexto:
{contexto}

Pregunta: {pregunta}

Responde de forma concisa en español. Si el contexto no contiene información suficiente, di exactamente "No tengo datos suficientes para responder a esta pregunta" y no inventes nada."""
    respuesta = llm.invoke(prompt).content
    return {"respuesta": respuesta, "fuentes": [r["texto"] for r in combinados], "pregunta": pregunta}

print("✅ Funciones auxiliares cargadas.")

✅ Funciones auxiliares cargadas.

## Ejercicios — contexto financiero

**Contexto:** Trabajas como Data Scientist en una consultora financiera.
Tu cliente quiere un sistema RAG que responda preguntas sobre su cartera
de activos financieros combinando informes narrativos en PDF con datos
estructurados de una base de datos SQLite de activos.

### Corpus — 3 informes financieros

In [6]:
informes_a = {
    "informe_anual_2025.pdf": (
        "Informe Anual 2025 — Cartera de Activos del Grupo Financiero M10",
        "El ejercicio 2025 ha sido favorable para la cartera de activos del Grupo "
        "Financiero M10, con una rentabilidad media ponderada del 7.2 %. Los activos "
        "de renta variable lideraron el crecimiento con un rendimiento conjunto del "
        "11.4 %, destacando la participación en el sector tecnológico (NASDAQ-100) "
        "que aportó 3.2 puntos porcentuales al resultado global. El IBEX 35 ETF "
        "cerró el año con una revalorización del 8.5 %, mientras que el S&P 500 "
        "Acciones Crecimiento rindió un 12.3 %, impulsado por el sector de IA.\n\n"
        "La renta fija mantuvo un comportamiento estable. Los Bonos Estatales a 10 "
        "años ofrecieron un rendimiento del 3.2 %, mientras que los Bonos Corporativos "
        "Grado de Inversión rindieron un 4.1 %, ligeramente por encima de la media "
        "del sector (3.8 %). Los Bonos Verdes, emitidos por primera vez en 2025, "
        "tuvieron una acogida excelente con una demanda 2.3 veces la oferta.\n\n"
        "En el segmento de inversión alternativa, los activos en capital privado "
        "(Private Equity Tech Fund) rindieron un 15.8 %, aunque con menor liquidez. "
        "Los activos en oro físico (GBP 100) se revalorizaron un 9.2 % en un entorno "
        "de incertidumbre geopolítica. El Fondo de Infraestructuras cumplió "
        "expectativas con un 6.7 %, ligeramente por debajo del objetivo del 7 %.\n\n"
        "El perfil de riesgo global de la cartera se mantiene en nivel moderado, con "
        "una exposición a riesgo alto del 18 % (dentro del límite del 20 %). La "
        "liquidez media de la cartera es alta, con un 65 % de los activos clasificados "
        "como de alta liquidez. Se recomienda mantener la asignación actual y revisar "
        "la exposición a renta fija soberana si el BCE confirma los recortes de tipos."
    ),
    "analisis_mercado_2026.pdf": (
        "Análisis de Mercado y Previsiones 2026 — División de Asset Management",
        "El panorama para 2026 se presenta con tipos de interés a la baja tras las "
        "señales del BCE de iniciar recortes graduales en el segundo semestre. Se "
        "espera que el tipo de referencia pase del 4.0 % al 3.25 % a cierre de 2026, "
        "lo que podría comprimir los rendimientos de la renta fija de nueva emisión "
        "pero impulsaría las valoraciones de la renta variable.\n\n"
        "Los sectores más prometedores para el próximo ejercicio son tecnología "
        "(especialmente IA y semiconductores), energías renovables y salud digital. "
        "Por el contrario, se recomienda infraponderar el sector inmobiliario "
        "comercial, donde los ratios de morosidad podrían alcanzar el 4.5 %, y el "
        "sector automotriz europeo, afectado por la competencia china.\n\n"
        "En el mercado de divisas, se espera un euro fortalecido frente al dólar "
        "(objetivo EUR/USD 1.12) y estable frente al franco suizo. La exposición a "
        "mercados emergentes (actualmente 800 M€ en divisas no EUR) requiere "
        "cobertura adicional, especialmente en real brasileño y rupia india.\n\n"
        "Las previsiones de crecimiento para la cartera modelo se sitúan en un rango "
        "del 5.5 % al 7.5 % para 2026, con un escenario base del 6.3 %. Los "
        "principales riesgos identificados son una posible recesión en EE.UU. "
        "(probabilidad estimada del 25 %), tensiones geopolíticas en el Este de "
        "Asia, y un repunte inflacionario que retrase los recortes del BCE."
    ),
    "informe_sostenibilidad_2025.pdf": (
        "Informe de Sostenibilidad e Inversión Responsable 2025",
        "El Grupo Financiero M10 ha integrado criterios ESG en el 78 % de su cartera "
        "de activos gestionados, superando el objetivo del 75 % fijado para 2025. "
        "Los activos clasificados como artículo 8 (SFDR) representan 3.200 M€ y los "
        "de artículo 9 (impacto positivo) alcanzan los 890 M€.\n\n"
        "Los Bonos Verdes emitidos en 2025 por valor de 500 M€ se destinaron "
        "íntegramente a proyectos de energía renovable (parques eólicos marinos en "
        "Galicia y plantas fotovoltaicas en Extremadura). La rentabilidad de estos "
        "bonos ha sido del 3.8 %, competitiva frente a bonos convencionales "
        "equivalentes. La demanda institucional superó en 2.3 veces la oferta.\n\n"
        "En el ámbito de exclusión, se han desinvertido 120 M€ en empresas con "
        "exposición a carbón térmico y 45 M€ en fabricantes de armamento controvertido. "
        "El fondo de inversión sostenible M10 ESG Leaders rindió un 9.4 % en 2025, "
        "superando en 1.2 puntos a su benchmark no ESG.\n\n"
        "La huella de carbono de la cartera (alcance 1 y 2) se redujo un 14 % "
        "respecto a 2024, hasta 85 tCO2e/M€ invertido. El objetivo para 2026 es "
        "alcanzar 75 tCO2e/M€. Se ha iniciado el proceso de medición del alcance 3 "
        "(cadena de valor) para reportar en el informe de 2026."
    ),
}
for nombre, (titulo, contenido) in informes_a.items():
    crear_pdf(nombre, titulo, contenido)
print(f"✅ {len(informes_a)} PDFs financieros generados.")

✅ 3 PDFs financieros generados.

### Base de datos SQLite — Activos financieros

In [7]:
DB_PATH_A = os.path.join(tempfile.mkdtemp(), "activos.db")
conn_a = sqlite3.connect(DB_PATH_A)
conn_a.execute("""
    CREATE TABLE activos_financieros (
        id INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        tipo TEXT NOT NULL,
        precio_actual REAL NOT NULL,
        rendimiento_pct REAL NOT NULL,
        riesgo TEXT NOT NULL,
        liquidez TEXT NOT NULL
    )
""")

activos_data = [
    (1,  "IBEX 35 ETF",               "Renta Variable",        45.20,  8.5,  "Medio",   "Alta"),
    (2,  "S&P 500 Acciones Crecim.",   "Renta Variable",       234.50, 12.3,  "Alto",    "Alta"),
    (3,  "NASDAQ-100 Index Fund",      "Renta Variable",       567.80, 15.2,  "Alto",    "Alta"),
    (4,  "Euro Stoxx 50 ETF",           "Renta Variable",       38.90,  7.1,  "Medio",   "Alta"),
    (5,  "Bono Estatal 10y",           "Renta Fija",           98.50,  3.2,  "Bajo",    "Alta"),
    (6,  "Bono Corporativo Grado Inv.","Renta Fija",          102.30,  4.1,  "Bajo",    "Media"),
    (7,  "Bono Verde 2025-2035",       "Renta Fija",          101.00,  3.8,  "Bajo",    "Media"),
    (8,  "Letras del Tesoro 6m",       "Renta Fija",           97.20,  2.5,  "Muy Bajo","Alta"),
    (9,  "Bono Subordinado Financiero","Renta Fija",           89.40,  5.6,  "Medio",   "Baja"),
    (10, "Private Equity Tech Fund",   "Inversión Alternativa",150.00, 15.8,  "Muy Alto","Baja"),
    (11, "Fondo Infraestructuras",     "Inversión Alternativa",125.00,  6.7,  "Medio",   "Baja"),
    (12, "Oro Físico GBP 100",         "Inversión Alternativa",189.20,  9.2,  "Medio",   "Alta"),
    (13, "Fondo Inmobiliario Comercial","Inversión Alternativa",95.00,   4.3,  "Alto",    "Baja"),
    (14, "M10 ESG Leaders Fund",       "Renta Variable",       112.40,  9.4,  "Medio",   "Media"),
    (15, "Deuda Corporativa High Yield","Renta Fija",           78.60,  7.8,  "Alto",    "Baja"),
    (16, "Fondo Monetario EUR",         "Renta Fija",          100.50,  2.1,  "Muy Bajo","Alta"),
    (17, "Futuros S&P 500",            "Derivados",            445.00,  0.0,  "Muy Alto","Alta"),
    (18, "ETF Energías Renovables",     "Renta Variable",       67.30, 11.2,  "Alto",    "Alta"),
    (19, "Deuda Emergente USD",        "Renta Fija",           82.10,  6.5,  "Alto",    "Media"),
    (20, "Cobertura Cambio EUR/USD",   "Derivados",            102.00,  0.0,  "Bajo",    "Alta"),
    (21, "Bonos Municipales España",   "Renta Fija",           99.80,  2.8,  "Bajo",    "Media"),
    (22, "Fondo Tecnología IA",        "Renta Variable",       178.30, 18.5,  "Muy Alto","Media"),
]

conn_a.executemany(
    "INSERT INTO activos_financieros VALUES (?, ?, ?, ?, ?, ?, ?)",
    activos_data
)
conn_a.commit()

df_activos = pd.DataFrame(activos_data, columns=[
    "id", "nombre", "tipo", "precio_actual", "rendimiento_pct", "riesgo", "liquidez"
])
df_activos["texto"] = df_activos.apply(
    lambda r: f"Activo: {r['nombre']}. Tipo: {r['tipo']}. Precio: {r['precio_actual']}€. "
              f"Rendimiento: {r['rendimiento_pct']}%. Riesgo: {r['riesgo']}. Liquidez: {r['liquidez']}.",
    axis=1
)

print(f"✅ {len(activos_data)} activos financieros en SQLite")
print(f"   Tipos de activo: {df_activos['tipo'].nunique()}")
print(f"   Rendimiento medio: {df_activos['rendimiento_pct'].mean():.1f}%")

✅ 22 activos financieros en SQLite
   Tipos de activo: 4
   Rendimiento medio: 7.1%

### Indexación dual

In [8]:
chunks_pdf_a = cargar_y_trocear_pdfs(CORPUS_DIR)
collection_pdf_a = client.get_or_create_collection(
    name="hpd5_finanzas_pdf", metadata={"hnsw:space": "cosine"}
)
indexar_chunks(collection_pdf_a, chunks_pdf_a, prefix="fin_")

collection_sql_a = client.get_or_create_collection(
    name="hpd5_finanzas_sql", metadata={"hnsw:space": "cosine"}
)
textos_sql_a = df_activos["texto"].tolist()
metas_sql_a = df_activos[["nombre", "tipo", "riesgo", "liquidez"]].to_dict("records")
indexar_textos(collection_sql_a, textos_sql_a, metas_sql_a, prefix="act_")

print(f"✅ PDFs indexados: {collection_pdf_a.count()} vectores")
print(f"✅ SQL indexados: {collection_sql_a.count()} vectores")

✅ PDFs indexados: 13 vectores
✅ SQL indexados: 22 vectores

#### Ejercicio 1 — Dataset de ground truth (2.5 ptos)

Crea un dataset de al menos 10 preguntas de evaluación con sus
respuestas ground truth (esperadas). Para cada pregunta, ejecuta tu RAG,
guarda la respuesta generada y los contextos recuperados. Usa este
dataset para evaluar la calidad de tu recuperación.

Asigna el resultado (lista de diccionarios con `question`, `answer`,
`contexts`, `ground_truth`) a la variable `eval1_dataset`.

In [9]:
preguntas_eval_a = [
    "¿Qué activos de renta variable tienen un rendimiento superior al 10%?",
    "¿Cuál es el rendimiento del Bono Estatal a 10 años?",
    "¿Qué activos de inversión alternativa tienen riesgo Muy Alto?",
    "¿Qué tipo de bono tuvo una demanda 2.3 veces la oferta en 2025?",
    "¿Cuáles son los sectores más prometedores para 2026 según el análisis?",
    "¿Qué activos tienen liquidez baja y riesgo alto o muy alto?",
    "¿Cómo ha evolucionado la integración ESG en la cartera?",
    "¿Qué activos de renta fija tienen un rendimiento superior al 4%?",
    "¿Cuál es la rentabilidad del Fondo de Infraestructuras y cómo se compara con su objetivo?",
    "¿Qué previsiones de crecimiento se esperan para la cartera en 2026?",
    "¿Qué activos tienen un precio superior a 150€ y riesgo medio o inferior?",
    "¿Qué se recomienda sobre la exposición a renta fija soberana?",
]

eval1_dataset = []
for pregunta in preguntas_eval_a:
    res = rag(pregunta, collection_pdf_a, collection_sql_a)
    eval1_dataset.append({
        "question": pregunta,
        "answer": res["respuesta"],
        "contexts": res["fuentes"],
        "ground_truth": "",  # ← completar con la respuesta esperada
    })

print(f"✅ eval1_dataset creado: {len(eval1_dataset)} preguntas")
for i, item in enumerate(eval1_dataset):
    print(f"  [{i+1}] {item['question'][:70]}...")

✅ eval1_dataset creado: 12 preguntas
  [1] ¿Qué activos de renta variable tienen un rendimiento superior al 10%?...
  [2] ¿Cuál es el rendimiento del Bono Estatal a 10 años?...
  [3] ¿Qué activos de inversión alternativa tienen riesgo Muy Alto?...
  [4] ¿Qué tipo de bono tuvo una demanda 2.3 veces la oferta en 2025?...
  [5] ¿Cuáles son los sectores más prometedores para 2026 según el análisis?...
  [6] ¿Qué activos tienen liquidez baja y riesgo alto o muy alto?...
  [7] ¿Cómo ha evolucionado la integración ESG en la cartera?...
  [8] ¿Qué activos de renta fija tienen un rendimiento superior al 4%?...
  [9] ¿Cuál es la rentabilidad del Fondo de Infraestructuras y cómo se compa...
  [10] ¿Qué previsiones de crecimiento se esperan para la cartera en 2026?...
  [11] ¿Qué activos tienen un precio superior a 150€ y riesgo medio o inferio...
  [12] ¿Qué se recomienda sobre la exposición a renta fija soberana?...

#### Ejercicio 2 — Evaluación RAGAS (2.5 ptos)

Usa RAGAS para evaluar tu pipeline con las métricas `faithfulness`,
`answer_relevancy` y `context_precision`. Genera un DataFrame con las
puntuaciones por pregunta y las medias globales.

Asigna el resultado (dict con `por_pregunta` y `medias`) a
`eval2_ragas`.

> **Por qué RAGAS necesita dos modelos**
>
> RAGAS no evalúa con una sola llamada: necesita un **juez (LLM)** y
> **embeddings**.
>
> - **Juez (LLM):** `juez_llm` lee respuestas y contextos y emite
>   veredictos. Lo usan `faithfulness`, `context_precision` y
>   `answer_relevancy`.
> - **Embeddings:** calculan similitudes entre vectores. Solo los usa
>   `answer_relevancy`, que compara tu pregunta con las preguntas que el
>   juez deduce de la respuesta. Si no se pasan, RAGAS intenta usar
>   OpenAI y falla con `OpenAIError: Missing credentials`. Aquí usamos
>   `LocalEmbeddings`, el mismo e5-small-v2 del pipeline, sin claves
>   extra.
>
> **Interpreta con cuidado las puntuaciones.** El juez es un modelo
> pequeño y puede puntuar 0 una respuesta honesta del tipo “No tengo
> datos suficientes para responder a esta pregunta”, aunque sea
> correcta: `faithfulness` la trata como una afirmación que el juez no
> deduce del contexto, y `answer_relevancy` pone 0 cuando el juez marca
> la respuesta como evasiva. Antes de cambiar el RAG por una puntuación
> 0, lee la respuesta y los contextos recuperados.

In [10]:
import json
from ragas import evaluate as ragas_evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from datasets import Dataset
from ragas.llms import LangchainLLMWrapper
from langchain_core.embeddings import Embeddings

# ─── Embeddings para answer_relevancy ───
# RAGAS necesita embeddings para comparar tu pregunta con las preguntas que el juez
# deduce de la respuesta. Si no se pasan, intenta usar OpenAI y falla sin clave.
# Usamos el mismo e5-small-v2 del pipeline (ya cargado en `model`), sin claves extra.
# e5 recomienda el prefijo "query: " en ambos lados para comparar textos entre sí
# (pregunta vs. pregunta). El pipeline no usa prefijos; para answer_relevancy basta
# con que los embeddings de preguntas sean coherentes entre sí.
class LocalEmbeddings(Embeddings):
    """Adapta SentenceTransformer a la interfaz de LangChain que RAGAS acepta."""

    def embed_documents(self, texts):
        return model.encode(["query: " + t for t in texts]).tolist()

    def embed_query(self, text):
        return model.encode(["query: " + text])[0].tolist()

# Llamus devuelve una sola respuesta por petición, y answer_relevancy pide 3 por defecto
# (strictness=3). Con strictness=1 genera una pregunta por respuesta: se evita el aviso
# "LLM returned 1 generations instead of requested 3", pero la métrica es más ruidosa
# y sus valores no son comparables con los de strictness=3.
answer_relevancy.strictness = 1

if eval1_dataset and all(item["ground_truth"] for item in eval1_dataset):
    dataset_hf = Dataset.from_dict({
        "question": [item["question"] for item in eval1_dataset],
        "answer": [item["answer"] for item in eval1_dataset],
        "contexts": [item["contexts"] for item in eval1_dataset],
        "ground_truth": [item["ground_truth"] for item in eval1_dataset],
    })

    evaluator_llm = LangchainLLMWrapper(juez_llm)
    result = ragas_evaluate(
        dataset=dataset_hf,
        metrics=[faithfulness, answer_relevancy, context_precision],
        llm=evaluator_llm,
        embeddings=LocalEmbeddings(),
    )

    df_ragas = result.to_pandas()
    eval2_ragas = {
        "por_pregunta": df_ragas[["faithfulness", "answer_relevancy", "context_precision"]].to_dict("records"),
        "medias": {
            "faithfulness": float(df_ragas["faithfulness"].mean()),
            "answer_relevancy": float(df_ragas["answer_relevancy"].mean()),
            "context_precision": float(df_ragas["context_precision"].mean()),
        }
    }

    print("📊 Resultados RAGAS (medias):")
    for metrica, valor in eval2_ragas["medias"].items():
        print(f"  {metrica}: {valor:.4f}")
    print(f"\nPuntuaciones por pregunta:")
    print(df_ragas[["faithfulness", "answer_relevancy", "context_precision"]])
else:
    eval2_ragas = {"error": "Completa los ground_truth en eval1_dataset primero."}
    print(eval2_ragas["error"])

Completa los ground_truth en eval1_dataset primero.

#### Ejercicio 3 — Iteración y mejora (2.5 ptos)

Modifica UN componente del pipeline (chunk_size, número de chunks
recuperados, prompt, modelo de embeddings, añade un re-ranker, etc.) y
vuelve a evaluar con RAGAS. Compara los resultados con la versión base y
explica por qué el cambio mejoró o empeoró las métricas.

Asigna el resultado (lista de diccionarios con `version`, `cambio`,
`faithfulness`, `answer_relevancy`, `context_precision`, `analisis`) a
`eval3_iteracion`.

In [11]:
eval3_iteracion = [
    # Completar con al menos 2 versiones (base + mejora)
    # {
    #     "version": "Base",
    #     "cambio": "CHUNK_SIZE=400, k_pdf=3, k_sql=3, sin re-ranker",
    #     "faithfulness": 0.0,
    #     "answer_relevancy": 0.0,
    #     "context_precision": 0.0,
    #     "analisis": "Configuración inicial del pipeline."
    # },
]
print(f"⚠️  Completa eval3_iteracion con al menos 2 versiones comparadas.")

⚠️  Completa eval3_iteracion con al menos 2 versiones comparadas.

#### Ejercicio 4 — Informe de evaluación (2.5 ptos)

Redacta un informe estructurado de evaluación que incluya: métricas
obtenidas, fortalezas y debilidades del pipeline, análisis de errores
(preguntas donde el RAG falló), impacto de la mejora implementada en E3,
y propuestas de mejora adicionales.

Asigna tu respuesta a `eval4_informe`.

In [12]:
eval4_informe = """Escribir aquí el informe estructurado. Por ejemplo:

## Resultados de evaluación
- Faithfulness medio: X.XXXX
- Answer Relevancy medio: X.XXXX
- Context Precision medio: X.XXXX

## Fortalezas
...

## Debilidades y errores identificados
...

## Impacto de la mejora
...

## Propuestas adicionales
..."""
print("✅ eval4_informe asignada. Completa el contenido.")

✅ eval4_informe asignada. Completa el contenido.

## Resumen de entregables

In [13]:
print(f"\nVariables de corrección:")
for var in ["eval1_dataset", "eval2_ragas", "eval3_iteracion", "eval4_informe"]:
    val = globals().get(var, None)
    status = "✅ definida" if val is not None else "⚠️ no definida"
    print(f"  {var}: {status}")


Variables de corrección:
  eval1_dataset: ✅ definida
  eval2_ragas: ✅ definida
  eval3_iteracion: ✅ definida
  eval4_informe: ✅ definida

# References